# 03 — Metamodel Joint Inference

Build the joint metamodel coupling all 4 surrogates and run Bayesian inference.

**Prerequisites**: Trained surrogates from notebook 02.

> **This notebook needs the `bayesian-metamodeling` framework.**
> It drives the `bayesmm` CLI, unlike the kinetic-segregation series under
> `notebooks/models/kinetic_segregation/`, which needs only numpy and the compiled
> model. If `bayesmm` is missing, install the framework
> (`pip install -e .` from the parent repo, or `pip install bayesian-metamodeling`)
> and restart the kernel.
>
> New here? Start at [`Tutorial_0_Start_Here.ipynb`](Tutorial_0_Start_Here.ipynb).

## Learning aims
- **Primary**: build the metamodel IR from coupled surrogates and sample the joint
  posterior.
- **Secondary scientific**: explain what a coupling asserts, and what "joint" buys you
  over four separate posteriors.

## What coupling means

Each partial model has its own posterior. Independently, their joint distribution is
just a product — nothing relates them. A **coupling** states that two variables in
different models are the same physical quantity, or are related by a known transform.

`specs/metamodel.tcr_signaling.json` couples `depletion_width_nm` with a
`gaussian_link` of sigma 10 nm: *these should agree, to within 10 nm*. That constraint
propagates — data that sharpens one model's posterior now sharpens its neighbours too.

**Requires `pymc`.** Sampling is the slow step; the draw counts here are deliberately
modest for interactive use, and production values are noted inline.

In [ ]:
import json
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find this repo, instead of assuming a fixed depth.

    The previous version walked up a fixed number of levels from the working
    directory, which silently assumed a submodule checkout and broke in a
    standalone clone or from any other directory. Searching for a landmark is
    robust to both.
    """
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "models" / "kinetic_segregation" / "CMakeLists.txt").is_file():
            return cand
    raise RuntimeError(f"could not locate the tcr_signaling repo above {here}")


ROOT = find_repo_root()
SPECS = ROOT / "specs"
print(f"repo root: {ROOT}")

# These notebooks drive the bayesian-metamodeling CLI. Report clearly if absent.
HAVE_BAYESMM = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "--version"], capture_output=True, text=True
).returncode == 0
print("bayesmm:", "available" if HAVE_BAYESMM else "NOT INSTALLED -- see the banner above")


## Build metamodel IR

In [ ]:
r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "meta", "build", str(META_SPEC)],
                   cwd=str(ROOT), capture_output=True, text=True)
print("Return code:", r.returncode)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)

## Sample from the joint posterior

In [ ]:
r = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "meta", "sample", str(META_SPEC), "--draws", "2000", "--tune", "1000"],
    cwd=str(ROOT), capture_output=True, text=True
)
print("Return code:", r.returncode)
print(r.stdout[:500])
if r.returncode != 0:
    print("STDERR:", r.stderr[:500])

## Inspect posterior samples

In [ ]:
r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "meta", "list"], cwd=str(ROOT),
                   capture_output=True, text=True)
print(r.stdout)

## Final check

In [ ]:
assert ROOT.is_dir()
print("[NB03 self-check OK]")